# Multi-Agent Framework Landscape

Once you move beyond a hand-rolled orchestrator, teams adopt **multi-agent frameworks** to avoid reinventing state management, message routing, retries, and observability. The ecosystem is crowded — **LangGraph**, **AutoGen**, **CrewAI**, and others each encode different **orchestration philosophies**.

This notebook maps the landscape without locking you into any library:

1. **What every framework solves** — shared primitives beneath the branding.
2. **Comparison dimensions** — control flow, state, communication, agent definition.
3. **Profiles** of major frameworks — strengths, trade-offs, sweet spots.
4. **Plain-Python adapters** that mirror each framework's core model on the same **ReleaseFlow** task.
5. **A selection guide** — which philosophy fits which product shape.

Everything runs offline. Optional cells show how to swap in real SDK calls when libraries are installed. No prior notebooks are required.

In [ ]:
"""
ReleaseFlow — shared scenario for framework comparison demos.
Goal: produce a v2.4.0 customer release announcement.
"""

import json
import os
import re
import uuid
from dataclasses import dataclass, field
from datetime import datetime, timezone
from enum import Enum
from typing import Any, Callable

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "sk-proj-placeholder-replace-with-your-real-key")
os.environ.setdefault("OPENAI_API_KEY", OPENAI_API_KEY)


def pretty(obj: Any) -> str:
    return json.dumps(obj, indent=2, default=str)


CHANGELOG: list[dict[str, str]] = [
    {"version": "2.4.0", "item": "Added bulk export endpoint for reports"},
    {"version": "2.4.0", "item": "Fixed session timeout on mobile browsers"},
    {"version": "2.4.0", "item": "Deprecated legacy v1 webhook format"},
]

RUNBOOKS: list[dict[str, str]] = [
    {"id": "comm-201", "title": "Customer communication standards",
     "text": "Lead with user benefit, list breaking changes separately, include support channel #releases."},
]

STYLE_GUIDE = {
    "must_include": ["version number", "breaking changes section", "support channel"],
    "max_words": 200,
}

print(f"ReleaseFlow corpus: {len(CHANGELOG)} changelog items")

---

## 1. The Problem Frameworks Solve

Hand-rolled multi-agent code works for demos. At scale you repeatedly build:

| Primitive | Without a framework | Framework provides |
|-----------|---------------------|------------------|
| **State** | Ad-hoc dicts passed around | Typed shared state, reducers, checkpoints |
| **Routing** | `if/else` spaghetti | Graph edges, supervisors, speaker selection |
| **Messages** | Custom log lists | Standard message schemas, thread history |
| **Retries** | Copy-paste per workflow | Built-in policies, human-in-the-loop hooks |
| **Persistence** | Roll your own | Thread IDs, resume from checkpoint |
| **Observability** | Print debugging | Trace spans, step metadata |

Frameworks are **opinionated orchestration layers** on top of LLM + tool APIs. The LLM is still the reasoning engine; the framework owns **who runs when** and **what state they see**.

---

## 2. Landscape at a Glance

| Framework | Primary metaphor | Control flow owner | Typical sweet spot |
|-----------|------------------|--------------------|--------------------|
| **LangGraph** | State graph (nodes + edges) | Engineer defines graph; optional agent loops inside nodes | Auditable DAGs, cyclic agent↔tool loops, production guardrails |
| **AutoGen** | Conversational agents | GroupChatManager or user proxy selects speaker | Research chat, human-in-loop coding, multi-turn debate |
| **CrewAI** | Role-based crew executing tasks | Crew `Process` (sequential, hierarchical) | Business workflows with clear roles and task lists |
| **LangChain Agents** | Single agent + tools (building block) | Model tool loop; often wrapped by LangGraph | Tool-calling assistants; usually not full MAS alone |
| **Semantic Kernel** | Plugins + planners (Microsoft stack) | Planner generates steps; connectors to enterprise | .NET / Azure shops, plugin ecosystems |
| **Plain Python** | Your orchestrator | You | Prototypes, tight control, learning |

No framework wins every task. Teams often use **LangGraph for production graphs** and borrow **patterns** from AutoGen (group chat) or CrewAI (role/task decomposition) conceptually.

---

## 3. Comparison Dimensions

Evaluate any framework on these axes:

| Dimension | Question to ask |
|-----------|-----------------|
| **Control flow** | Graph, conversation, or task queue? |
| **State model** | Single shared object, per-agent memory, or message-only? |
| **Agent definition** | Class, config dict, or YAML role card? |
| **Communication** | Broadcast chat, directed handoff, or blackboard artifacts? |
| **Human-in-the-loop** | First-class pause/resume or custom code? |
| **Tool integration** | Native tool nodes, function calling, MCP? |
| **Persistence** | Checkpoints, thread store, none? |
| **Vendor coupling** | LangChain ecosystem, Microsoft, standalone? |

In [ ]:
class ControlModel(str, Enum):
    GRAPH = "graph"
    CONVERSATION = "conversation"
    TASK_CREW = "task_crew"
    PLANNER = "planner"
    CUSTOM = "custom"


@dataclass
class FrameworkProfile:
    name: str
    control_model: ControlModel
    state_style: str
    agent_definition: str
    strengths: list[str]
    tradeoffs: list[str]
    best_for: list[str]
    learning_curve: str  # low | medium | high


FRAMEWORKS: list[FrameworkProfile] = [
    FrameworkProfile(
        name="LangGraph",
        control_model=ControlModel.GRAPH,
        state_style="TypedDict + reducers (e.g. append messages)",
        agent_definition="Graph nodes wrapping LLM + ToolNode",
        strengths=["Explicit DAG", "Cyclic agent loops", "Checkpointing", "Conditional routing"],
        tradeoffs=["LangChain coupling", "Boilerplate for simple flows", "Graph mental model required"],
        best_for=["Production workflows", "Compliance audits", "Hybrid workflow+agent"],
        learning_curve="medium",
    ),
    FrameworkProfile(
        name="AutoGen",
        control_model=ControlModel.CONVERSATION,
        state_style="Chat message history per agent / group",
        agent_definition="ConversableAgent with system_message + tools",
        strengths=["Natural multi-turn chat", "Human proxy pattern", "Flexible speaker selection"],
        tradeoffs=["Chat transcripts grow fast", "Routing can be opaque", "Cost control needs discipline"],
        best_for=["Research sessions", "Coding assistants", "Debate / review by discussion"],
        learning_curve="medium",
    ),
    FrameworkProfile(
        name="CrewAI",
        control_model=ControlModel.TASK_CREW,
        state_style="Task outputs chained as context",
        agent_definition="Agent(role, goal, backstory) + Task(description, agent)",
        strengths=["Readable role cards", "Fast to prototype crews", "Sequential/hierarchical processes"],
        tradeoffs=["Less fine-grained routing", "Opinionated abstractions", "Debugging across tasks"],
        best_for=["Marketing copy pipelines", "Research reports", "Role-play specialist teams"],
        learning_curve="low",
    ),
    FrameworkProfile(
        name="Plain Python",
        control_model=ControlModel.CUSTOM,
        state_style="Your dataclass / dict",
        agent_definition="Callable classes or functions",
        strengths=["Zero deps", "Full control", "Easy to test"],
        tradeoffs=["You build persistence, tracing, HITL", "Does not scale in complexity alone"],
        best_for=["Learning", "Tight SLAs", "Embedding in existing services"],
        learning_curve="low",
    ),
]

print(f"{'Framework':<14} {'Control':<14} {'Curve':<8} Best for")
print("-" * 70)
for f in FRAMEWORKS:
    print(f"{f.name:<14} {f.control_model.value:<14} {f.learning_curve:<8} {f.best_for[0]}")

---

## 4. Shared ReleaseFlow Agents

All framework adapters below run the **same three specialists** on the **same data** so comparisons are fair.

In [ ]:
@dataclass
class ReleaseState:
    goal: str
    artifacts: dict[str, Any] = field(default_factory=dict)
    messages: list[dict[str, str]] = field(default_factory=list)
    trace: list[str] = field(default_factory=list)

    def log(self, speaker: str, content: str) -> None:
        self.messages.append({"speaker": speaker, "content": content, "ts": datetime.now(timezone.utc).isoformat()})
        self.trace.append(f"{speaker}: {content[:60]}")


def researcher(state: ReleaseState) -> ReleaseState:
    facts = {"version": "2.4.0", "changes": [c["item"] for c in CHANGELOG], "standards": RUNBOOKS[0]["text"]}
    state.artifacts["research"] = facts
    state.log("researcher", f"Collected {len(facts['changes'])} changes for v{facts['version']}")
    return state


def writer(state: ReleaseState) -> ReleaseState:
    facts = state.artifacts.get("research", {})
    draft = (
        f"Release v{facts.get('version', '?')} is now available.\n\n"
        f"What's new: " + "; ".join(facts.get("changes", [])) + ".\n\n"
        f"Breaking changes: Legacy v1 webhook format is deprecated.\n\n"
        f"Questions? Contact #releases."
    )
    state.artifacts["draft"] = draft
    state.log("writer", f"Draft ready ({len(draft.split())} words)")
    return state


def critic(state: ReleaseState) -> ReleaseState:
    draft = state.artifacts.get("draft", "")
    issues = []
    if "v2.4" not in draft:
        issues.append("missing version")
    if "Breaking changes" not in draft:
        issues.append("missing breaking section")
    if "#releases" not in draft:
        issues.append("missing support channel")
    if len(draft.split()) > STYLE_GUIDE["max_words"]:
        issues.append("too long")
    approved = not issues
    state.artifacts["approved"] = approved
    state.log("critic", "APPROVED" if approved else f"REJECTED: {', '.join(issues)}")
    return state


AGENTS = {"researcher": researcher, "writer": writer, "critic": critic}
print("Shared agents:", list(AGENTS.keys()))

---

## 5. LangGraph Mental Model — State Graph

LangGraph models orchestration as a **directed graph**:

```
        START
          │
          ▼
     ┌──────────┐
     │ research │
     └────┬─────┘
          ▼
     ┌──────────┐
     │  write   │◄────┐
     └────┬─────┘     │ reject
          ▼           │
     ┌──────────┐     │
     │  review  │─────┘
     └────┬─────┘ approve
          ▼
         END
```

| LangGraph concept | Meaning |
|-------------------|--------|
| `StateGraph` | Workflow with typed shared state |
| `add_node` | Agent or function step |
| `add_conditional_edges` | Router function picks next node |
| `ToolNode` | Executes parallel tool calls |
| `checkpointer` | Persist state between runs |

In [ ]:
@dataclass
class LangGraphStyleAdapter:
    """Plain-Python mirror of a LangGraph StateGraph for ReleaseFlow."""

    name: str = "LangGraph-style"
    nodes: dict[str, Callable[[ReleaseState], ReleaseState]] = field(default_factory=lambda: dict(AGENTS))

    def route_after_review(self, state: ReleaseState) -> str:
        if state.artifacts.get("approved"):
            return "END"
        if state.artifacts.get("retry_count", 0) >= 1:
            return "END"
        state.artifacts["retry_count"] = state.artifacts.get("retry_count", 0) + 1
        return "write"

    def run(self, goal: str) -> ReleaseState:
        state = ReleaseState(goal=goal)
        graph_trace = ["START"]

        for node_name in ("researcher", "writer", "critic"):
            state = self.nodes[node_name](state)
            graph_trace.append(node_name)

        while True:
            nxt = self.route_after_review(state)
            graph_trace.append(f"route→{nxt}")
            if nxt == "END":
                break
            state = self.nodes[nxt](state)
            state = self.nodes["critic"](state)
            graph_trace.append("writer→critic")

        state.artifacts["graph_trace"] = graph_trace
        return state


lg = LangGraphStyleAdapter()
lg_result = lg.run("v2.4.0 announcement")
print("Graph path:", " → ".join(lg_result.artifacts["graph_trace"]))
print("Approved:", lg_result.artifacts.get("approved"))

---

## 6. AutoGen Mental Model — Group Conversation

AutoGen centers on **agents that chat**. A **GroupChatManager** (or similar selector) picks the next speaker from a roster.

```
UserProxy: "Produce release announcement"
    Researcher: "Here are the v2.4.0 changes..."
    Writer: "Draft: Release v2.4.0 is now available..."
    Critic: "APPROVED"
    Manager: terminate
```

| AutoGen concept | Meaning |
|-----------------|--------|
| `ConversableAgent` | LLM + system prompt + optional tools |
| `UserProxyAgent` | Human or scripted initiator |
| `GroupChat` | Shared transcript, round-robin or selector |
| `GroupChatManager` | Decides next speaker |

In [ ]:
class AutoGenStyleAdapter:
    """Plain-Python mirror of AutoGen GroupChat speaker selection."""

    name = "AutoGen-style"
    roster = ["researcher", "writer", "critic"]

    def select_speaker(self, turn: int, state: ReleaseState) -> str | None:
        if turn < len(self.roster):
            return self.roster[turn]
        if not state.artifacts.get("approved") and state.artifacts.get("chat_rounds", 0) < 1:
            state.artifacts["chat_rounds"] = state.artifacts.get("chat_rounds", 0) + 1
            return "writer"  # manager sends back to writer for revision
        return None  # terminate

    def run(self, goal: str) -> ReleaseState:
        state = ReleaseState(goal=goal)
        state.log("user_proxy", goal)
        turn = 0
        chat_trace = []

        while True:
            speaker = self.select_speaker(turn, state)
            if speaker is None:
                chat_trace.append("manager: TERMINATE")
                break
            chat_trace.append(f"manager selects: {speaker}")
            state = AGENTS[speaker](state)
            turn += 1
            if speaker == "writer" and turn > len(self.roster):
                state = AGENTS["critic"](state)
                chat_trace.append("manager selects: critic")
                turn += 1

        state.artifacts["chat_trace"] = chat_trace
        return state


ag = AutoGenStyleAdapter()
ag_result = ag.run("v2.4.0 announcement")
print("Chat trace:")
for line in ag_result.artifacts["chat_trace"]:
    print(f"  {line}")
print("Approved:", ag_result.artifacts.get("approved"))

---

## 7. CrewAI Mental Model — Roles and Tasks

CrewAI organizes work as a **crew** of agents executing a **task list** with a defined **process**.

```python
# Conceptual CrewAI shape (not executed here)
researcher = Agent(role="Researcher", goal="Gather release facts", ...)
writer     = Agent(role="Writer", goal="Draft announcement", ...)
critic     = Agent(role="Critic", goal="Review against style guide", ...)

tasks = [
    Task(description="Research v2.4.0 changes", agent=researcher),
    Task(description="Write customer email", agent=writer, context=[task1]),
    Task(description="Review draft", agent=critic, context=[task2]),
]
crew = Crew(agents=[...], tasks=tasks, process=Process.sequential)
```

| CrewAI concept | Meaning |
|----------------|--------|
| `Agent` | Role card: role, goal, backstory |
| `Task` | Unit of work with description + assigned agent |
| `Crew` | Team + task list + process type |
| `Process.sequential` | Tasks run in order; output feeds context |

In [ ]:
@dataclass
class CrewTask:
    description: str
    agent_name: str
    context_from: list[str] = field(default_factory=list)


@dataclass
class CrewRole:
    name: str
    role: str
    goal: str


class CrewAIStyleAdapter:
    """Plain-Python mirror of CrewAI sequential Process."""

    name = "CrewAI-style"

    def __init__(self) -> None:
        self.roles = [
            CrewRole("researcher", "Senior Research Analyst", "Gather accurate release facts"),
            CrewRole("writer", "Technical Writer", "Draft customer-facing announcement"),
            CrewRole("critic", "Editor", "Enforce style guide compliance"),
        ]
        self.tasks = [
            CrewTask("Research v2.4.0 changelog and comm standards", "researcher"),
            CrewTask("Write release email from research", "writer", context_from=["research"]),
            CrewTask("Review draft against style guide", "critic", context_from=["draft"]),
        ]

    def run(self, goal: str) -> ReleaseState:
        state = ReleaseState(goal=goal)
        task_outputs: dict[str, str] = {}
        crew_trace = []

        for i, task in enumerate(self.tasks, start=1):
            role = next(r for r in self.roles if r.name == task.agent_name)
            crew_trace.append(f"Task {i}: {task.description} → {role.role}")
            state = AGENTS[task.agent_name](state)
            if task.agent_name == "researcher":
                task_outputs["research"] = pretty(state.artifacts.get("research", {}))
            elif task.agent_name == "writer":
                task_outputs["draft"] = state.artifacts.get("draft", "")[:80]
            elif task.agent_name == "critic":
                task_outputs["verdict"] = state.messages[-1]["content"]

        state.artifacts["crew_trace"] = crew_trace
        state.artifacts["task_outputs"] = task_outputs
        return state


crew = CrewAIStyleAdapter()
crew_result = crew.run("v2.4.0 announcement")
print("Crew task trace:")
for line in crew_result.artifacts["crew_trace"]:
    print(f"  {line}")
print("Approved:", crew_result.artifacts.get("approved"))

---

## 8. Plain Python Baseline

Before adopting a framework, implement once in plain Python. You will understand what the framework is automating.

In [ ]:
class PlainPythonAdapter:
    name = "Plain Python"

    def run(self, goal: str) -> ReleaseState:
        state = ReleaseState(goal=goal)
        for name in ("researcher", "writer", "critic"):
            state = AGENTS[name](state)
        return state


plain = PlainPythonAdapter()
plain_result = plain.run("v2.4.0 announcement")
print(f"Steps: {len(plain_result.trace)} | Approved: {plain_result.artifacts.get('approved')}")

---

## 9. Side-by-Side Run Comparison

Same goal, four orchestration philosophies:

In [ ]:
ADAPTERS = [
    PlainPythonAdapter(),
    LangGraphStyleAdapter(),
    AutoGenStyleAdapter(),
    CrewAIStyleAdapter(),
]

GOAL = "Produce v2.4.0 customer release announcement"

print(f"{'Adapter':<18} {'Approved':<10} {'Msgs':<6} {'Control model'}")
print("-" * 60)
for adapter in ADAPTERS:
    result = adapter.run(GOAL)
    profile = next((f for f in FRAMEWORKS if f.name.startswith(adapter.name.split("-")[0]) or f.name == adapter.name), None)
    control = profile.control_model.value if profile else "custom"
    if adapter.name == "LangGraph-style":
        control = "graph"
    elif adapter.name == "AutoGen-style":
        control = "conversation"
    elif adapter.name == "CrewAI-style":
        control = "task_crew"
    print(f"{adapter.name:<18} {str(result.artifacts.get('approved')):<10} {len(result.messages):<6} {control}")

---

## 10. Feature Matrix

Qualitative scoring: ● = strong, ○ = partial, — = weak / not primary focus.

In [ ]:
FEATURES = ["Explicit graph", "Group chat", "Role/task cards", "Checkpointing", "Tool nodes", "HITL hooks", "Low deps"]

MATRIX: dict[str, dict[str, str]] = {
    "LangGraph":       {"Explicit graph": "●", "Group chat": "○", "Role/task cards": "—", "Checkpointing": "●", "Tool nodes": "●", "HITL hooks": "●", "Low deps": "—"},
    "AutoGen":         {"Explicit graph": "○", "Group chat": "●", "Role/task cards": "—", "Checkpointing": "○", "Tool nodes": "○", "HITL hooks": "●", "Low deps": "○"},
    "CrewAI":          {"Explicit graph": "○", "Group chat": "○", "Role/task cards": "●", "Checkpointing": "—", "Tool nodes": "○", "HITL hooks": "○", "Low deps": "○"},
    "Plain Python":    {"Explicit graph": "●", "Group chat": "—", "Role/task cards": "—", "Checkpointing": "—", "Tool nodes": "—", "HITL hooks": "—", "Low deps": "●"},
}

header = f"{'Feature':<18}" + "".join(f"{fw:<12}" for fw in MATRIX)
print(header)
print("-" * len(header))
for feat in FEATURES:
    row = f"{feat:<18}" + "".join(f"{MATRIX[fw].get(feat, '—'):<12}" for fw in MATRIX)
    print(row)

---

## 11. Framework Selection Guide

```
Need auditable DAG with branches/loops?
  YES → LangGraph (or plain graph + later migrate)
  NO ↓

Is the UX a multi-agent conversation?
  YES → AutoGen-style group chat
  NO ↓

Is work a clear list of role-owned tasks?
  YES → CrewAI-style crew
  NO ↓

Prototype or embedded micro-orchestrator?
  YES → Plain Python until complexity forces a framework
```

| Your situation | Lean toward |
|----------------|-------------|
| Compliance, fixed checkpoints, prod SLAs | LangGraph |
| Research chat, coding with human proxy | AutoGen |
| Marketing / ops pipelines with roles | CrewAI |
| Learning, unit tests, minimal deps | Plain Python |
| Already deep in LangChain ecosystem | LangGraph |
| Azure / .NET enterprise plugins | Semantic Kernel |

In [ ]:
def recommend_framework(
    *,
    needs_audit_graph: bool = False,
    conversation_ux: bool = False,
    role_task_pipeline: bool = False,
    minimal_deps: bool = False,
) -> str:
    if minimal_deps and not needs_audit_graph:
        return "Plain Python"
    if needs_audit_graph:
        return "LangGraph"
    if conversation_ux:
        return "AutoGen"
    if role_task_pipeline:
        return "CrewAI"
    return "Plain Python (start simple)"


SCENARIOS = [
    ("Production incident workflow with HITL", dict(needs_audit_graph=True)),
    ("Research brainstorm UI", dict(conversation_ux=True)),
    ("Content crew: researcher + writer + SEO", dict(role_task_pipeline=True)),
    ("Embedded agent in existing FastAPI service", dict(minimal_deps=True)),
]

for label, kwargs in SCENARIOS:
    print(f"{label:<45} → {recommend_framework(**kwargs)}")

---

## 12. What Frameworks Share Under the Hood

Different APIs, same skeleton:

```
┌─────────────────────────────────────────────────────────┐
│                    Orchestration layer                   │
│  (graph edges | speaker selector | task queue)          │
├─────────────────────────────────────────────────────────┤
│  Agent A          Agent B          Agent C               │
│  (prompt+tools)   (prompt+tools)   (prompt+tools)       │
├─────────────────────────────────────────────────────────┤
│  Shared state / message bus / task context               │
├─────────────────────────────────────────────────────────┤
│  LLM provider API          Tool registry / MCP           │
└─────────────────────────────────────────────────────────┘
```

Learning one framework deeply transfers — you are learning **orchestration patterns**, not irreplaceable syntax.

---

## 13. Migration Path — Plain Python First

| Stage | What you have | When to upgrade |
|-------|---------------|-----------------|
| **1. Prototype** | `for step in steps: state = step(state)` | Works for linear demos |
| **2. Complexity** | Branches, retries, parallel fan-out | Graph framework reduces bugs |
| **3. Production** | Need checkpoints, HITL, tracing | LangGraph or managed platform |
| **4. Product UX** | Users see agent chat | AutoGen-style conversation layer |

**Tip:** Keep agent **implementations** (researcher, writer, critic) separate from **orchestration** so you can swap adapters without rewriting specialists.

In [ ]:
class OrchestratorProtocol:
    """Interface any framework adapter should implement."""
    name: str

    def run(self, goal: str) -> ReleaseState:
        raise NotImplementedError


def benchmark_adapters(adapters: list, goal: str) -> list[dict[str, Any]]:
    rows = []
    for a in adapters:
        t0 = datetime.now(timezone.utc)
        result = a.run(goal)
        ms = (datetime.now(timezone.utc) - t0).total_seconds() * 1000
        rows.append({
            "adapter": a.name,
            "approved": result.artifacts.get("approved"),
            "messages": len(result.messages),
            "ms": round(ms, 2),
        })
    return rows


bench = benchmark_adapters(ADAPTERS, GOAL)
print(pretty(bench))

---

## 14. When Not to Use a Framework

| Situation | Why skip frameworks |
|-----------|---------------------|
| Single agent + 3 tools | Framework overhead exceeds benefit |
| Hard real-time latency budget | Extra layers add ms |
| Orchestration already in your platform | Step Functions, Temporal, Airflow + one LLM step |
| Team lacks graph/debug skills | Plain code easier to onboard |
| Vendor lock-in concerns | Plain Python + provider SDK only |

---

## 15. Optional — Detect Installed SDKs

This cell checks whether common libraries are importable. Nothing breaks if they are missing.

In [ ]:
def check_import(package: str) -> str:
    try:
        __import__(package)
        return "installed"
    except ImportError:
        return "not installed"


SDK_STATUS = {
    "langgraph": check_import("langgraph"),
    "langchain_core": check_import("langchain_core"),
    "autogen": check_import("autogen"),
    "crewai": check_import("crewai"),
    "openai": check_import("openai"),
}

print("SDK availability (informational only):")
for pkg, status in SDK_STATUS.items():
    print(f"  {pkg:<16} {status}")
print("\nAll demos above run without any of these packages.")

---

## 16. Conceptual SDK Mapping — ReleaseFlow in Real Frameworks

When you do install libraries, the ReleaseFlow pipeline maps roughly as follows:

**LangGraph**
```python
# builder.add_node("research", research_node)
# builder.add_node("write", write_node)
# builder.add_node("review", review_node)
# builder.add_conditional_edges("review", route, {"write": "write", "END": END})
```

**AutoGen**
```python
# groupchat = GroupChat(agents=[researcher, writer, critic], messages=[], max_round=6)
# manager = GroupChatManager(groupchat=groupchat, llm_config=...)
```

**CrewAI**
```python
# crew = Crew(agents=[...], tasks=[...], process=Process.sequential)
# result = crew.kickoff(inputs={"goal": "v2.4.0 announcement"})
```

Your plain-Python adapters are **executable specifications** of these shapes.

---

## 17. Quiz

1. What three control models distinguish LangGraph, AutoGen, and CrewAI?
2. Name two primitives every multi-agent framework provides beyond raw LLM calls.
3. When would you choose group chat over a state graph?
4. Why implement plain Python before adopting a framework?
5. What does `Process.sequential` mean in CrewAI?

---

## 18. Summary

| Framework | Core metaphor | Best when |
|-----------|---------------|-----------|
| **LangGraph** | State graph with conditional edges | Auditable production workflows |
| **AutoGen** | Multi-agent conversation + manager | Research chat, HITL coding |
| **CrewAI** | Crew of roles executing tasks | Role-based content/ops pipelines |
| **Plain Python** | Your orchestrator | Learning, prototypes, tight embedding |

| Takeaway | Detail |
|----------|--------|
| Same agents, different routers | Researcher/writer/critic unchanged across adapters |
| Compare on dimensions | Control, state, communication — not hype |
| Start simple | Migrate when branches, checkpoints, or HITL demand it |
| Patterns transfer | Graph, chat, and task-queue ideas recur across vendors |

Pick the orchestration philosophy that matches how your **team thinks about the product** — graphs for compliance pipelines, chat for exploratory UX, task crews for role-owned deliverables.